# Week 4 Exercise: Python -> High-Performance C++ Converter

Build a model-selectable converter that ports Python to optimized C++ using frontier models, streams generation in Gradio, and benchmarks each model's performance in temporary session state.

In [1]:
# Imports

import io
import os
import re
import sys
import time
import tempfile
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple

import gradio as gr
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
# Environment + model clients

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

openai_client = OpenAI(api_key=openai_api_key, base_url="https://openrouter.ai/api/v1") if openai_api_key else None
anthropic_client = OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1/") if anthropic_api_key else None
gemini_client = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/") if google_api_key else None
grok_client = OpenAI(api_key=grok_api_key, base_url="https://api.x.ai/v1") if grok_api_key else None
groq_client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1") if groq_api_key else None
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1") if openrouter_api_key else None

MODEL_REGISTRY: Dict[str, Dict] = {
    "gpt-5": {"client": openai_client, "supports_reasoning": True},
    "claude-sonnet-4-5-20250929": {"client": anthropic_client, "supports_reasoning": False},
    "grok-4": {"client": grok_client, "supports_reasoning": False},
    "gemini-2.5-pro": {"client": gemini_client, "supports_reasoning": False},
    "openai/gpt-oss-120b": {"client": groq_client, "supports_reasoning": False},
    "qwen/qwen3-coder-30b-a3b-instruct": {"client": openrouter_client, "supports_reasoning": False},
}

AVAILABLE_MODELS = [name for name, cfg in MODEL_REGISTRY.items() if cfg["client"] is not None]

if not AVAILABLE_MODELS:
    raise RuntimeError(
        "No model clients available. Add at least one API key in your .env (OPENAI_API_KEY, ANTHROPIC_API_KEY, GOOGLE_API_KEY, GROK_API_KEY, GROQ_API_KEY, OPENROUTER_API_KEY)."
    )

print("Available models:")
for m in AVAILABLE_MODELS:
    print("-", m)

Available models:
- gpt-5
- gemini-2.5-pro
- qwen/qwen3-coder-30b-a3b-instruct


In [4]:
# Prompt + conversion helpers

SYSTEM_PROMPT = """
You convert Python programs into optimized, high-performance C++17.
Return ONLY valid C++ code, no prose.
Preserve the exact output behavior of the Python program.
Prefer performance-focused choices (tight loops, static typing, efficient memory usage) while staying correct.
""".strip()


def user_prompt_for(python_code: str) -> str:
    return f"""
Port the following Python code to optimized C++17.
Your response will be compiled with clang++ and executed for benchmarking.
Return only compilable C++ code.

Python code:
```python
{python_code}
```
""".strip()


def messages_for(python_code: str) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt_for(python_code)},
    ]


def clean_cpp_reply(text: str) -> str:
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:cpp|c\+\+)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    return cleaned.strip()


def _create_completion(model_name: str, python_code: str, stream: bool = False):
    cfg = MODEL_REGISTRY[model_name]
    client = cfg["client"]
    kwargs = {
        "model": model_name,
        "messages": messages_for(python_code),
        "stream": stream,
    }
    if cfg.get("supports_reasoning"):
        kwargs["reasoning_effort"] = "high"
    return client.chat.completions.create(**kwargs)


def convert_python_to_cpp(model_name: str, python_code: str) -> str:
    response = _create_completion(model_name, python_code, stream=False)
    message = response.choices[0].message.content or ""
    return clean_cpp_reply(message)


def stream_python_to_cpp(model_name: str, python_code: str):
    accumulated = ""
    yield accumulated, f"Streaming from {model_name}..."

    try:
        stream = _create_completion(model_name, python_code, stream=True)
        for chunk in stream:
            if not chunk.choices:
                continue
            delta = chunk.choices[0].delta
            piece = getattr(delta, "content", None)

            if isinstance(piece, list):
                piece = "".join(
                    part.get("text", "") if isinstance(part, dict) else str(part)
                    for part in piece
                )

            if piece:
                accumulated += piece
                yield clean_cpp_reply(accumulated), f"Streaming from {model_name}..."

        final_cpp = clean_cpp_reply(accumulated)
        yield final_cpp, f"Conversion complete using {model_name}."
    except Exception as e:
        yield accumulated, f"Conversion failed for {model_name}: {e}"

In [5]:
# Execution + benchmark helpers


def run_python_script(python_code: str) -> Tuple[str, float, str]:
    with tempfile.TemporaryDirectory() as tmp_dir:
        script_path = Path(tmp_dir) / "workload.py"
        script_path.write_text(python_code, encoding="utf-8")

        start = time.perf_counter()
        proc = subprocess.run(
            [sys.executable, str(script_path)],
            text=True,
            capture_output=True,
        )
        elapsed = time.perf_counter() - start

        if proc.returncode != 0:
            return "", elapsed, proc.stderr.strip()

        return proc.stdout, elapsed, ""


def compile_cpp(cpp_code: str, build_dir: Path) -> Path:
    source_path = build_dir / "main.cpp"
    binary_path = build_dir / "main"
    source_path.write_text(cpp_code, encoding="utf-8")

    compile_command = [
        "clang++",
        "-std=c++17",
        "-Ofast",
        "-DNDEBUG",
        str(source_path),
        "-o",
        str(binary_path),
    ]
    proc = subprocess.run(compile_command, text=True, capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip() or "C++ compilation failed.")

    return binary_path


def run_binary(binary_path: Path) -> Tuple[str, float, str]:
    start = time.perf_counter()
    proc = subprocess.run([str(binary_path)], text=True, capture_output=True)
    elapsed = time.perf_counter() - start

    if proc.returncode != 0:
        return "", elapsed, proc.stderr.strip()

    return proc.stdout, elapsed, ""


def benchmark_model(model_name: str, python_code: str, runs: int) -> Dict:
    python_out, py_t, py_err = run_python_script(python_code)
    if py_err:
        return {
            "model": model_name,
            "status": "python_error",
            "python_time_s": py_t,
            "cpp_time_s": None,
            "speedup_x": None,
            "output_match": False,
            "error": py_err,
        }

    cpp_code = convert_python_to_cpp(model_name, python_code)

    try:
        with tempfile.TemporaryDirectory(prefix="cpp_bench_") as tmp_dir:
            build_dir = Path(tmp_dir)
            binary_path = compile_cpp(cpp_code, build_dir)

            run_times = []
            cpp_out = ""
            for _ in range(runs):
                cpp_out, run_t, run_err = run_binary(binary_path)
                if run_err:
                    return {
                        "model": model_name,
                        "status": "runtime_error",
                        "python_time_s": py_t,
                        "cpp_time_s": None,
                        "speedup_x": None,
                        "output_match": False,
                        "error": run_err,
                    }
                run_times.append(run_t)

    except Exception as e:
        return {
            "model": model_name,
            "status": "compile_error",
            "python_time_s": py_t,
            "cpp_time_s": None,
            "speedup_x": None,
            "output_match": False,
            "error": str(e),
        }

    cpp_t = sum(run_times) / len(run_times)
    speedup = (py_t / cpp_t) if cpp_t > 0 else None
    match = python_out.strip() == cpp_out.strip()

    return {
        "model": model_name,
        "status": "ok",
        "python_time_s": round(py_t, 6),
        "cpp_time_s": round(cpp_t, 6),
        "speedup_x": round(speedup, 2) if speedup else None,
        "output_match": match,
        "error": "" if match else "Output mismatch",
        "generated_cpp": cpp_code,
    }


def state_to_dataframe(state: Dict[str, Dict]) -> pd.DataFrame:
    if not state:
        return pd.DataFrame(
            columns=["model", "status", "python_time_s", "cpp_time_s", "speedup_x", "output_match", "error"]
        )

    rows = [
        {
            "model": v.get("model", k),
            "status": v.get("status"),
            "python_time_s": v.get("python_time_s"),
            "cpp_time_s": v.get("cpp_time_s"),
            "speedup_x": v.get("speedup_x"),
            "output_match": v.get("output_match"),
            "error": v.get("error", ""),
        }
        for k, v in state.items()
    ]

    df = pd.DataFrame(rows)
    if "speedup_x" in df.columns:
        df = df.sort_values(by="speedup_x", ascending=False, na_position="last")
    return df.reset_index(drop=True)

In [6]:
# Sample workload (edit this during experiments)

sample_python = r'''
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= (1 / j)
        j = i * param1 + param2
        result += (1 / j)
    return result

start_time = time.time()
result = calculate(5_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
'''.strip()

In [7]:
# Gradio app: model selection, streaming, and temporary benchmark state


def benchmark_selected_models(selected_models: List[str], python_code: str, runs: int, session_state: Dict[str, Dict]):
    if session_state is None:
        session_state = {}

    if not selected_models:
        df = state_to_dataframe(session_state)
        return df, session_state, "Select at least one model for benchmarking."

    for model_name in selected_models:
        try:
            result = benchmark_model(model_name, python_code, runs)
        except Exception as e:
            result = {
                "model": model_name,
                "status": "failed",
                "python_time_s": None,
                "cpp_time_s": None,
                "speedup_x": None,
                "output_match": False,
                "error": str(e),
            }
        session_state[model_name] = result

    df = state_to_dataframe(session_state)
    return df, session_state, "Benchmark complete. State is stored for this live session only."


def clear_benchmark_state():
    empty_state = {}
    return state_to_dataframe(empty_state), empty_state, "Benchmark state cleared."


with gr.Blocks(title="Python -> Optimized C++ Converter") as demo:
    gr.Markdown("## Python -> Optimized C++ (Frontier Models)\nChoose a model, stream conversion, then benchmark selected models.")

    session_state = gr.State({})

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_code = gr.Code(label="Python code", language="python", value=sample_python, lines=22)
        with gr.Column(scale=6):
            cpp_code = gr.Code(label="Generated C++", language="cpp", value="", lines=22)

    with gr.Row():
        convert_model = gr.Dropdown(choices=AVAILABLE_MODELS, value=AVAILABLE_MODELS[0], label="Conversion model")
        convert_btn = gr.Button("Convert")

    convert_status = gr.Textbox(label="Conversion status", value="Idle")

    convert_btn.click(
        fn=stream_python_to_cpp,
        inputs=[convert_model, python_code],
        outputs=[cpp_code, convert_status],
    )

    gr.Markdown("### Benchmark models")
    with gr.Row():
        bench_models = gr.CheckboxGroup(choices=AVAILABLE_MODELS, value=AVAILABLE_MODELS[: min(3, len(AVAILABLE_MODELS))], label="Models to benchmark")
        runs = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="Binary runs per model (average)")

    with gr.Row():
        benchmark_btn = gr.Button("Run benchmark")
        clear_btn = gr.Button("Clear temporary state")

    benchmark_table = gr.Dataframe(
        headers=["model", "status", "python_time_s", "cpp_time_s", "speedup_x", "output_match", "error"],
        datatype=["str", "str", "number", "number", "number", "bool", "str"],
        row_count=(1, "dynamic"),
        col_count=(7, "fixed"),
        label="Per-model benchmark state",
    )
    benchmark_status = gr.Textbox(label="Benchmark status", value="Idle")

    benchmark_btn.click(
        fn=benchmark_selected_models,
        inputs=[bench_models, python_code, runs, session_state],
        outputs=[benchmark_table, session_state, benchmark_status],
    )

    clear_btn.click(
        fn=clear_benchmark_state,
        inputs=[],
        outputs=[benchmark_table, session_state, benchmark_status],
    )


demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


### Notes
- The benchmark table is a **temporary per-session state** in `gr.State`, reset when the app restarts.
- `output_match` compares Python stdout and C++ stdout after trimming whitespace.
- If `clang++` is missing, install Xcode command line tools on macOS (`xcode-select --install`).